# HopSkipJump Attack on All Models

This notebook assumes you already have the following helper functions available
in the environment (import them from your own modules before running this):

- `run_logreg("CSVs/newDataset.csv")`
- `run_neuralnet("CSVs/newDataset.csv")`
- `run_randomforest("CSVs/newDataset.csv")`
- `run_svm("CSVs/newDataset.csv")`
- `run_xgboost("CSVs/newDataset.csv")`

Each function is expected to return a tuple:

```python
(model, X_test, y_test)
```

where:
- `model` is a trained classifier (pipeline or estimator)
- `X_test` is the test feature matrix (pandas DataFrame or numpy array)
- `y_test` is the corresponding true labels

The notebook then applies the HopSkipJump attack from ART to each model and
reports clean vs adversarial accuracy.


In [12]:
# Imports

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score

# Adversarial Robustness Toolbox (ART)
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump

# IMPORTANT:
# Make sure you import your run_* helpers here, for example:
#
from MachineLearning.LogReg.LogisticRegression_ML import run_best_model as run_logreg
from MachineLearning.NeuralNetworks.NeuralNet_ML import run_best_model as run_neuralnet
from MachineLearning.RandomForest.RandomForest_ML import run_best_model as run_randomforest
from MachineLearning.SVM.SVM_ML import run_best_model as run_svm
from MachineLearning.XGBoost.XGBoost_ML import run_best_model as run_xgboost
#
# or import your wrapper functions that already call those.


import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but StandardScaler was fitted with feature names",
    category=UserWarning,
)



In [13]:
# Set seed for reproducibility

SEED = 42
np.random.seed(SEED)

In [14]:
# Run all models on the same dataset path
# Assumes each function returns (model, X_test, y_test)

DATA_PATH = "CSVs/newDataset.csv"

model_runs = {}

print("Running Logistic Regression...")
logreg_model, logreg_X_test, logreg_y_test = run_logreg(DATA_PATH)
model_runs["LogisticRegression"] = (logreg_model, logreg_X_test, logreg_y_test)

print("Running Neural Net...")
nn_model, nn_X_test, nn_y_test = run_neuralnet(DATA_PATH)
model_runs["NeuralNet"] = (nn_model, nn_X_test, nn_y_test)

print("Running Random Forest...")
rf_model, rf_X_test, rf_y_test = run_randomforest(DATA_PATH)
model_runs["RandomForest"] = (rf_model, rf_X_test, rf_y_test)

print("Running SVM...")
svm_model, svm_X_test, svm_y_test = run_svm(DATA_PATH)
model_runs["SVM"] = (svm_model, svm_X_test, svm_y_test)

print("Running XGBoost...")
xgb_model, xgb_X_test, xgb_y_test = run_xgboost(DATA_PATH)
model_runs["XGBoost"] = (xgb_model, xgb_X_test, xgb_y_test)

print("\nSummary of collected models:")
for name, (model, X_test, y_test) in model_runs.items():
    print(f" - {name}: model={type(model)}, X_test shape={getattr(X_test, 'shape', None)}")



Running Logistic Regression...
Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.936)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1352
           1       0.77      0.83      0.80       347

    accuracy                           0.92      1699
   macro avg       0.87      0.89      0.87      1699
weighted avg       0.92      0.92      0.92      1699


Confusion Matrix:
         Pred 0  Pred 1
True 0    1268      84
True 1      58     289

AUC: 0.936

=== All results and summaries saved successfully ===
Running Neural Net...
Using parameters: {'hidden_layer_sizes': '128,64', 'alpha': 0.0001, 'learning_rate_

C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:183: UserWarning: [21:21:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [15]:
hsj_kwargs = dict(
    max_iter=1,
    max_eval=2,
    init_eval=2,
    targeted=False,
)


results = {}

for name, (model, X_test, y_test) in model_runs.items():
    print("\n=== Attacking model:", name, "===")

    if hasattr(X_test, "to_numpy"):
        X = X_test.to_numpy().astype(np.float32)
    else:
        X = np.array(X_test, dtype=np.float32)

    if hasattr(y_test, "to_numpy"):
        y = y_test.to_numpy()
    else:
        y = np.array(y_test)

    X_min = X.min(axis=0)
    X_max = X.max(axis=0)
    clip_values = (X_min, X_max)

    art_clf = SklearnClassifier(model=model, clip_values=clip_values)
    attack = HopSkipJump(classifier=art_clf, **hsj_kwargs)

    X_adv = attack.generate(x=X)

    pred_clean = model.predict(X)
    pred_adv = model.predict(X_adv)

    clean_acc = (pred_clean == y).mean()
    adv_acc = (pred_adv == y).mean()

    print(f"{name} clean acc: {clean_acc:.4f}")
    print(f"{name} adv   acc: {adv_acc:.4f}")

    results[name] = dict(clean_acc=float(clean_acc), adv_acc=float(adv_acc))

results_df = pd.DataFrame(results).T
results_df



=== Attacking model: LogisticRegression ===


HopSkipJump: 100%|██████████| 1699/1699 [00:11<00:00, 141.61it/s]


LogisticRegression clean acc: 0.9164
LogisticRegression adv   acc: 0.2042

=== Attacking model: NeuralNet ===


HopSkipJump: 100%|██████████| 1699/1699 [00:11<00:00, 143.50it/s]


NeuralNet clean acc: 0.9459
NeuralNet adv   acc: 0.2031

=== Attacking model: RandomForest ===


HopSkipJump: 100%|██████████| 1699/1699 [12:54<00:00,  2.19it/s]


RandomForest clean acc: 0.9453
RandomForest adv   acc: 0.0571

=== Attacking model: SVM ===


HopSkipJump: 100%|██████████| 1911/1911 [00:10<00:00, 178.32it/s]

SVM clean acc: 0.9299
SVM adv   acc: 0.1868

=== Attacking model: XGBoost ===


TypeError: Model is not an sklearn model. Received '<class 'xgboost.sklearn.XGBClassifier'>'